# Import dependencies

In [30]:
from ipyleaflet import Map, basemaps, GeoJSON, Choropleth, FullScreenControl, ZoomControl, WidgetControl
from ipywidgets import HTML, Button, Dropdown, IntSlider, HBox, VBox, Label, Layout
from branca.colormap import linear
import plotly.graph_objects as go
import geopandas as gpd
import pandas as pd
import json

# Loading Mapping Data

In [31]:
#LSOA to MSOA mapping table
LSOA_to_MSOA = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["LSOA21CD", "LSOA21NM", "MSOA21CD", "MSOA21NM"]).drop_duplicates()

#MSOA to LAD mapping table
MSOA_to_LAD = pd.read_csv("OA_to_LSOA_to_MSOA_to_LAD_(December 2021).csv", usecols=["MSOA21CD", "MSOA21NM", "LAD22CD", "LAD22NM"]).drop_duplicates().rename(columns={"LAD22CD":"LAD21CD", "LAD22NM":"LAD21NM"})

#LAD to PFA mapping table
LAD_to_PFA = pd.read_excel("LAD_to_PFA_(December 2021).xlsx", usecols=["LAD21CD", "LAD21NM", "PFA21CD", "PFA21NM"]).drop_duplicates()

#Merging the previous two mapping tables
MSOA_to_PFA = pd.merge(MSOA_to_LAD, LAD_to_PFA, on="LAD21CD", how="inner").drop(["LAD21CD", "LAD21NM_x", "LAD21NM_y"], axis=1)

# Loading Geospatial Data

In [32]:
#Boundary data of PFAs
with open("PFA_(2021)_BGC.geojson",'r') as pfa:
    pfa_data = json.load(pfa)
    
#Boundary data of MSOAs
with open("MSOA_(2021)_BGC.geojson",'r') as msoa:
    msoa_data = json.load(msoa)

#Rename id values in the geojsons to the PFA or MSOA codes for easier handling
for i in pfa_data['features']:
    i['id'] = i['properties']['PFA21CD']
for i in msoa_data['features']:
    i['id'] = i['properties']['MSOA21CD']

#English MSOA demographics data
ENG_MSOA_demog_data = pd.read_csv("msoa_data/ENG_MSOA_demographic_data.csv")

# Aggregating LSOA demographic data to MSOA level

In [33]:
'''
#English LSOA demographics data
ENG_LSOA_demog_data = pd.read_csv("msoa_data/lsoa_demographics_from_db_file.csv")

#English MSOA demographics data
#Initial dummy row
data = {"MSOA21CD":[0],
        "pop":[0],
        "working_pop":[0],
        "elderly_pop":[0],
        "child_pop":[0],
        "econ_score":[0],
        "infrastructure_score":[0],
        "health_score":[0]
        }
ENG_MSOA_demog_data = pd.DataFrame(data)

#Aggregate LSOA data to MSOA level by summing population values and averaging IMD scores 
for msoa in msoa_data["features"]:
    #Only use English MSOAs due to IMD data
    if not msoa["id"].startswith("W"):
        msoa_pop = 0
        msoa_working_pop = 0
        msoa_elderly_pop = 0
        msoa_child_pop = 0
        msoa_econ_score = 0
        msoa_infrastructure_score = 0
        msoa_health_score = 0
        #Compute MSOA values using each LSOA in the MSOA
        LSOAs_in_msoa = LSOA_to_MSOA.loc[LSOA_to_MSOA["MSOA21CD"] == msoa["id"]]["LSOA21CD"]
        for lsoa in LSOAs_in_msoa:
            LSOA_row = ENG_LSOA_demog_data.loc[ENG_LSOA_demog_data["lsoa_code"] == lsoa]
            msoa_pop += LSOA_row["pop"].values[0]
            msoa_working_pop += msoa_pop * LSOA_row["percent_working"].values[0]
            msoa_elderly_pop += msoa_pop * LSOA_row["percent_old"].values[0]
            msoa_child_pop += msoa_pop * LSOA_row["percent_child"].values[0]
            msoa_econ_score += LSOA_row["econ_score"].values[0]
            msoa_infrastructure_score += LSOA_row["infrastructure_score"].values[0]
            msoa_health_score += LSOA_row["health_score"].values[0]
        msoa_econ_score /= LSOAs_in_msoa.size
        msoa_infrastructure_score /= LSOAs_in_msoa.size
        msoa_health_score /= LSOAs_in_msoa.size
        #Package computed values
        data = {"MSOA21CD":[msoa["id"]],
                "pop":[msoa_pop],
                "working_pop":[msoa_working_pop],
                "elderly_pop":[msoa_elderly_pop],
                "child_pop":[msoa_child_pop],
                "econ_score":[msoa_econ_score],
                "infrastructure_score":[msoa_infrastructure_score],
                "health_score":[msoa_health_score]
               }
        msoa_dataframe = pd.DataFrame(data)
        #Merge the newly computed MSOA into the dataframe
        ENG_MSOA_demog_data = pd.concat([ENG_MSOA_demog_data, msoa_dataframe])
        
#Remove dummy row, fix indexing and export to CSV
ENG_MSOA_demog_data.index = range(ENG_MSOA_demog_data.shape[0])
ENG_MSOA_demog_data.drop(index=0, inplace=True)
ENG_MSOA_demog_data.index = range(ENG_MSOA_demog_data.shape[0])
ENG_MSOA_demog_data.to_csv("msoa_data/ENG_MSOA_demographic_data.csv", index=False)
'''
''

''

# Set up choropleth data

In [34]:
'''
pfa_resources =
msoa_crime_predictions =
msoa_allocations =
'''
ENG_msoa_pop = ENG_MSOA_demog_data[["MSOA21CD", "pop"]].set_index("MSOA21CD").to_dict()

# Create basic PFA and MSOA layers

In [35]:
pfa_layer = GeoJSON(data=pfa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)
msoa_layer = GeoJSON(data=msoa_data, 
    style={'color': 'black', 'fillColor': '#E0D071', 'opacity':0.25, 'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6},
    hover_style={'fillColor': '#b08a3e' , 'fillOpacity': 0.8}
)

'''
pfa_layer = GeoJSON(data=pfa_data,
    style={'color': 'black', 'opacity': 0.25},
    hover_style={'opacity': 0.5}
)

msoa_layer = GeoJSON(data=msoa_data,
    style={'color': 'black', 'opacity': 0.25},
    hover_style={'opacity': 0.5}
)

pfa_choropleth = Choropleth(
    geo_data=pfa_data,
    choro_data= pfa_resources,
    key_on="id",
    colormap=linear.YlGnBu_07,
    style={'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6}
)

msoa_choropleth = Choropleth(
    geo_data=msoa_data
    choro_data= msoa_crime_predictions,
    key_on="id"
    colormap=linear.Oranges_08,
    style={'weight':1.9, 'dashArray':'2', 'fillOpacity':0.6}
)
'''

'\npfa_layer = GeoJSON(data=pfa_data,\n    style={\'color\': \'black\', \'opacity\': 0.25},\n    hover_style={\'opacity\': 0.5}\n)\n\nmsoa_layer = GeoJSON(data=msoa_data,\n    style={\'color\': \'black\', \'opacity\': 0.25},\n    hover_style={\'opacity\': 0.5}\n)\n\npfa_choropleth = Choropleth(\n    geo_data=pfa_data,\n    choro_data= pfa_resources,\n    key_on="id",\n    colormap=linear.YlGnBu_07,\n    style={\'weight\':1.9, \'dashArray\':\'2\', \'fillOpacity\':0.6}\n)\n\nmsoa_choropleth = Choropleth(\n    geo_data=msoa_data\n    choro_data= msoa_crime_predictions,\n    key_on="id"\n    colormap=linear.Oranges_08,\n    style={\'weight\':1.9, \'dashArray\':\'2\', \'fillOpacity\':0.6}\n)\n'

# Creating the interactive visualisation

In [54]:
plt.ioff()

#----------------#
#Create basic map#
#----------------#

center = [53,-2.5]
zoom = 7
m = Map(basemap=basemaps.CartoDB.Positron, center=center, zoom=zoom, zoom_control=False)

#----------------------------------#
#Create interactive control widgets#
#----------------------------------#

#Set up PFA information display HTML
html_pfa = HTML('''<h3><b>Hover over a Police Force!</b></h3>''')
html_pfa.layout.margin = '0px 20px 20px 20px'
pfa_control = WidgetControl(widget=html_pfa, position='topright')

#Set up MSOA information display HTMLs
html_msoa_demog = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa_demog.layout.margin = '0px 20px 20px 20px'
html_msoa_crime = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa_crime.layout.margin = '0px 20px 20px 20px'
html_msoa_alloc = HTML('''<h3><b>Hover over an MSOA!</b></h3>''')
html_msoa_crime.layout.margin = '0px 20px 20px 20px'

#Set up buttons that change what MSOA information is displayed
crime_button = Button(
    description='Crime Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Crime Information",
)
allocation_button = Button(
    description='Allocation Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Allocation Information",
)
demographic_button = Button(
    description='Demographic Info',
    disabled=False,
    button_style='',
    tooltip="Displays Data for Demographic Information",
)
#Package the buttons
info_buttons = HBox([crime_button, allocation_button, demographic_button])

#Set up prediction month selection
msoa_month_predict_select = IntSlider(
    value=1,
    min=1,
    max=3,
    step=1,
    description='',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)
msoa_month_predict_select.layout.width = "50%"

#Set up button that refreshes the crime data when a new month is selected
display_month_crime_button = Button(
    description="Display",
    disabled=False,
    button_style='',
    tooltip="Displays Data for the Selected Month",
    icon="check"
)

#Package all crime and allocation information widgets
msoa_crime_box = VBox([VBox([Label(value="How many month(s) ahead do you wish to see crime information?"), HBox([msoa_month_predict_select, display_month_crime_button])]), html_msoa_crime])
msoa_alloc_box = VBox([VBox([Label(value="How many month(s) ahead do you wish to see allocation information?"), HBox([msoa_month_predict_select, display_month_crime_button])]), html_msoa_alloc])

#Package all MSOA information widgets
msoa_info = VBox([info_buttons, msoa_crime_box])
msoa_info_control = WidgetControl(widget=msoa_info, position='topright')

#Set up button to return to PFA view in MSOA view
return_to_pfa_button = Button(
    description="Return to PFA view",
    disabled=False,
    button_style='',
    tooltip="Return to PFA view",
    icon="arrow-left"
)
return_control = WidgetControl(widget=return_to_pfa_button, position='topright')

data = ENG_MSOA_demog_data.loc[ENG_MSOA_demog_data["MSOA21CD"] == "E02000001"][["working_pop", "elderly_pop", "child_pop"]].squeeze(axis=0)
pie = go.Pie(labels=["Working Age", "Elderly", "Children"],
             values=data.values,
             textinfo="label+percent"
            )
fig = go.FigureWidget(data=[pie],
                      layout=go.Layout(width=500,
                                       height=500, 
                                       title="Population Distribution",
                                       title_font_size=20,
                                      )
                     )
plot_control = WidgetControl(widget=fig, position="bottomleft")

#-------------------------------------------------#
#Implement functions to update interactive widgets#
#-------------------------------------------------#

#Define PFA HTML update function
def update_pfa(**kwargs):
    html_pfa.value = '''
                 <h3><b>Police Force: </b>{}</h3>
                 <p>PFA Code: {}</p>
                 <p>Available Neighbourhood Police (Headcount / FTEs): <b> / </b></p>
                 <ul>
                    <li>Of which Police Officers: <b> / </b></li>
                    <li>Of which PCSOs: <b> / </b></li>
                 <ul>
                 '''.format(kwargs['properties']['PFA21NM'], kwargs['properties']['PFA21CD'])
    
#Define MSOA HTML update function
def update_msoa(**kwargs):
    
    #IF-ELIF CLAUSES DECIDING WHAT MONTH PREDICTION/ALLOCATION TO DISPLAY
    '''
    if msoa_month_predict_select.value == 1:
        #change prediction month to 1
        #change allocation month to 1
        msoa_choropleth.choro_data=
    elif msoa_month_predict_select.value == 2:
        #change prediction month to 2
        #change allocation month to 2
        msoa_choropleth.choro_data=
    elif msoa_month_predict_select.value == 3:
        #change prediction month to 3
        #change allocation month to 3
        msoa_choropleth.choro_data=
    '''

    fig.data[0].values=ENG_MSOA_demog_data.loc[ENG_MSOA_demog_data["MSOA21CD"] == kwargs["id"]][["working_pop", "elderly_pop", "child_pop"]].squeeze(axis=0).values

    html_msoa_crime.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Predicted Crime Counts per Category:</p>
        <ul>
            <li>Anti-Social Behaviour: <b> </b></li>
            <li>Violence: <b> </b></li>
            <li>Property Theft: <b> </b></li>
            <li>Disruptive Crimes: <b> </b></li>
            <li>Criminal Damage and Arson: <b> </b></li>
            <li>Vehicle Crime: <b> </b></li>
            <li>Burglary: <b> </b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])
    html_msoa_alloc.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Suggested Allocation of Neighbourhood Police per Group:</p>
        <ul>
            <li>Neighbourhood Problem-Solving and Reassurance: <b> </b></li>
            <li>Acquisitive and Place-Based Prevention: <b> </b></li>
            <li>Harm, Disruption and Enforcement: <b> </b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])
    html_msoa_demog.value = '''
        <h3><b>MSOA: </b>{}</h3>
        <p>MSOA Code: {}</p>
        <p>Population: <b> </b></p>
        <ul>
            <li>Of which working age: <b> </b></li>
            <li>Of which children: <b> </b></li>
            <li>Of which elderly: <b> </b></li>
        </ul>
        <p>Average Demographic Scores across Constituient LSOAs:</p>
        <ul>
            <li>Economy Score: <b> </b></li>
            <li>Infrastructure Score: <b> </b></li>
            <li>Health Score: <b> </b></li>
        </ul>
        '''.format(kwargs['properties']['MSOA21NM'], kwargs['properties']['MSOA21CD'])

#Define MSOA info change button function
def msoa_info_change(button_instance):
    if button_instance.description == 'Crime Info':
        msoa_info = VBox([info_buttons, msoa_crime_box])
        #msoa_choropleth.choro_data =
    elif button_instance.description == 'Allocation Info':
        msoa_info = VBox([info_buttons, msoa_alloc_box])
        #msoa_choropleth.choro_data =
    elif button_instance.description == 'Demographic Info':
        msoa_info = VBox([info_buttons, html_msoa_demog])
        #msoa_choropleth.choro_data =
    msoa_info_control.widget=msoa_info

#Define return button function
def back_to_pfa(button_instance):
    m.substitute(msoa_layer, pfa_layer)
    #m.substitute(msoa_choropleth, pfa_choropleth)
    m.remove(msoa_info_control)
    m.remove(return_control)
    m.center =  center
    m.zoom = zoom   
    
#Define function to update the map when a PFA is clicked
def whenClicked_PFA(**kwargs):
    MSOAs_in_PFA = MSOA_to_PFA.loc[MSOA_to_PFA["PFA21CD"] == kwargs["properties"]["PFA21CD"]]["MSOA21CD"]
    MSOAs_json = {
                  "type": "FeatureCollection", 
                  "crs": {"type": "name", "properties": {"name": "EPSG:4326"}},
                  "features": []
                 }
    for i in msoa_data["features"]:
        for j in MSOAs_in_PFA:
            if i["properties"]["MSOA21CD"] == j:
                MSOAs_json["features"].append(i)
                break
    msoa_layer.data = MSOAs_json
    #msoa_choropleth.geo_data = MSOAs_json
    m.substitute(pfa_layer, msoa_layer)
    #m.substitute(pfa_choropleth, msoa_choropleth)
    m.add(msoa_info_control)
    m.add(return_control)
    m.center =  [kwargs['properties']['LAT'], kwargs['properties']['LONG']]
    m.zoom = 9.25
    


#----------------------------------#
#Attach update functions to widgets#
#----------------------------------#

pfa_layer.on_click(whenClicked_PFA)
pfa_layer.on_hover(update_pfa)

msoa_layer.on_hover(update_msoa)

return_to_pfa_button.on_click(back_to_pfa)
display_month_crime_button.on_click(update_msoa)
crime_button.on_click(msoa_info_change)
allocation_button.on_click(msoa_info_change)
demographic_button.on_click(msoa_info_change)

#----------------------------------------#
#Add default elements and display the map#
#----------------------------------------#

m.add(pfa_layer)
m.add(ZoomControl(position="bottomright"))
m.add(FullScreenControl(position="bottomright"))
m.add(pfa_control)
m.add(plot_control)

m

Map(center=[53, -2.5], controls=(AttributionControl(options=['position', 'prefix'], position='bottomright'), Z…